In [1]:
%load_ext autoreload
%autoreload 2

%autosave 30

Autosaving every 30 seconds


In [3]:
import os

os.environ["AWS_PROFILE"] = "mermaid-core"

## CoralNet annotations

Merged CoralNet point annotations are stored as a single Parquet file on S3 (see `CoralNetDataset` in `mermaidseg/datasets/dataset.py`). The block below loads that file with Ibis and inspects the S3 object layout for source `4957` (images are expected under `coralnet-public-images/s4957/images/`).

In [4]:
import ibis
import pandas as pd
from great_tables import GT
from ibis import _

# Defaults match CoralNetDataset in mermaidseg/datasets/dataset.py
CORALNET_SOURCE_BUCKET = "dev-datamermaid-sm-sources"
CORALNET_ANNOTATIONS_PARQUET = "coralnet_annotations_30112025.parquet"
CORALNET_ANNOTATIONS_URI = f"s3://{CORALNET_SOURCE_BUCKET}/{CORALNET_ANNOTATIONS_PARQUET}"
CORALNET_SOURCE_S3_PREFIX = "coralnet-public-images"
EXAMPLE_SOURCE_ID = 5086
AUDIT_PATH = "coralnet_source_audit_20260423.parquet"
AUDIT_URI = f"s3://{CORALNET_SOURCE_BUCKET}/dev/{AUDIT_PATH}"
BEST_SOURCES = f"s3://{CORALNET_SOURCE_BUCKET}/dev/baselines/coralnet_best_sources - coralnet_best_sources.csv"
ibis.options.interactive = True

con = ibis.duckdb.connect()
con.raw_sql(
    """
    INSTALL httpfs;
    LOAD httpfs;
    """
)
# Second statement: DuckDB validates the credential chain here — if this fails, fix AWS first
# (skill preflight: `aws sts get-caller-identity`, or rerun the notebook cell that runs check_aws_session).
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
s3_audit = con.read_parquet(AUDIT_URI)

In [6]:
# Recompute is_complete so that it also includes that n_annotations > 0
s3_audit = s3_audit.mutate(
    is_complete=_.has_images_folder & _.has_annotations_csv & _.has_image_list_csv & _.has_annotations_csv & (_.n_annotations > 0)
)

In [7]:
summary_data = {
    "Metric": [
        "Total sources",
        "Complete sources",
        "Sources with images folder",
        "Sources with annotations.csv",
        "Sources with image_list.csv",
        "Sources with labelset.csv",
        "Sources with metadata.csv",
        "Sources with image count match",
        "Sources with empty annotations.csv",
        "Sources with empty image_list.csv",
        "Sources with empty labelset.csv",
        "Sources with empty metadata.csv",
    ],
    "Count": [
        s3_audit.count().execute(),
        s3_audit.filter(_.is_complete).count().execute(),
        s3_audit.filter(_.has_images_folder).count().execute(),
        s3_audit.filter(_.has_annotations_csv).count().execute(),
        s3_audit.filter(_.has_image_list_csv).count().execute(),
        s3_audit.filter(_.has_labelset_csv).count().execute(),
        s3_audit.filter(_.has_metadata_csv).count().execute(),
        s3_audit.filter(_.image_count_match).count().execute(),
        s3_audit.filter(_.annotations_empty).count().execute(),
        s3_audit.filter(_.image_list_empty).count().execute(),
        s3_audit.filter(_.labelset_empty).count().execute(),
        s3_audit.filter(_.metadata_empty).count().execute(),
    ],
}
summary_df = pd.DataFrame(summary_data)
total = summary_df.loc[0, "Count"]
summary_df["Percentage"] = (summary_df["Count"] / total * 100).round(1)

In [8]:
GT(summary_df).tab_header(title="CoralNet Source Audit Summary")

GT(_tbl_data=                                Metric  Count  Percentage
0                        Total sources   1509       100.0
1                     Complete sources    812        53.8
2           Sources with images folder   1461        96.8
3         Sources with annotations.csv    898        59.5
4          Sources with image_list.csv    897        59.4
5            Sources with labelset.csv    902        59.8
6            Sources with metadata.csv    612        40.6
7       Sources with image count match    903        59.8
8   Sources with empty annotations.csv     79         5.2
9    Sources with empty image_list.csv      0         0.0
10     Sources with empty labelset.csv      0         0.0
11     Sources with empty metadata.csv      0         0.0, _body=<great_tables._gt_data.Body object at 0x1a57b6ff0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='right', column_width=None), ColInfo(var='Percentage', type=<ColInfoTypeEnum.default: 1>, column_label='Percentage', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1a45ca9c0>, _spanners=Spanners([]), _heading=Heading(title='CoralNet Source Audit Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1a5ebeb10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1a5ebeb40>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1a5ebe870>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=Tru

In [9]:
totals = s3_audit.aggregate(
    total_images_s3=s3_audit.n_images_s3.sum(),
    total_images_csv=s3_audit.n_images_csv.sum(),
    total_annotations=s3_audit.n_annotations.sum(),
    total_unique_annotated=s3_audit.n_unique_images_annotated.sum(),
    total_labels=s3_audit.n_labels.sum(),
    total_metadata_rows=s3_audit.n_metadata_rows.sum(),
).execute()

totals_df = totals.T.reset_index()
totals_df.columns = ["Metric", "Value"]

In [10]:
GT(totals_df).tab_header(title="Total Resource Counts").fmt_number(columns=["Value"], sep_mark=",", decimals=0)

GT(_tbl_data=                   Metric       Value
0         total_images_s3   1101071.0
1        total_images_csv    507349.0
2       total_annotations  26812746.0
3  total_unique_annotated    903805.0
4            total_labels     42188.0
5     total_metadata_rows      2133.0, _body=<great_tables._gt_data.Body object at 0x108bc61b0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1a5fb23c0>, _spanners=Spanners([]), _heading=Heading(title='Total Resource Counts', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1a5fcede0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1a5fce4e0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1a5fcedb0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1a5fcf620>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='he

In [11]:
cn_annotations_raw = con.read_parquet(CORALNET_ANNOTATIONS_URI)
cn_for_source = cn_annotations_raw.filter(cn_annotations_raw.source_id == EXAMPLE_SOURCE_ID)

anno_summary = cn_for_source.aggregate(
    n_points=cn_for_source.count(),
    n_images=cn_for_source.image_id.nunique(),
).execute()

In [12]:
cn_annotations_raw

┏━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┓
┃ source_id ┃ image_id ┃ row   ┃ col   ┃ coralnet_id ┃
┡━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━┩
│ int64     │ string   │ int64 │ int64 │ int64       │
├───────────┼──────────┼───────┼───────┼─────────────┤
│        23 │ 12805    │   735 │  1008 │         112 │
│        23 │ 12805    │   663 │  1682 │         106 │
│        23 │ 12805    │   955 │  1737 │         106 │
│        23 │ 12805    │  1034 │  1431 │         105 │
│        23 │ 12805    │   851 │  2036 │         106 │
│        23 │ 12805    │   833 │  2038 │         100 │
│        23 │ 12805    │  1235 │  1158 │         111 │
│        23 │ 12805    │  1294 │  1312 │         105 │
│        23 │ 12805    │  1194 │  1319 │         105 │
│        23 │ 12805    │  1498 │  3096 │         105 │
│         … │ …        │     … │     … │           … │
└───────────┴──────────┴───────┴───────┴─────────────┘

In [13]:
anno_summary

,n_points,n_images
0,20200,1000


In [14]:
sources_p = cn_annotations_raw.select(_.source_id).distinct()

In [15]:
LIST_OF_SOURCES = [4957, 5086, 5444, 5847, 6521, 6537, 6687, 6699]

In [16]:
sources_p.filter(sources_p.source_id.isin(LIST_OF_SOURCES)).execute()

,source_id
0,6537
1,6687
2,5444
3,5086
4,5847
5,6699
6,6521


## Compare to existing pre-computed parquet file from the past

In [17]:
audit_complete = s3_audit.filter(_.is_complete).select(source_id=_.source_id)
legacy_sources = cn_annotations_raw.select(_.source_id).distinct()

in_audit_not_legacy = audit_complete.anti_join(legacy_sources, "source_id")
in_legacy_not_audit = legacy_sources.anti_join(audit_complete, "source_id")

print(f"Complete sources in audit: {audit_complete.count().execute()}")
print(f"Distinct sources in legacy parquet: {legacy_sources.count().execute()}")
print(f"In audit (complete) but NOT in legacy: {in_audit_not_legacy.count().execute()}")
print(f"In legacy but NOT complete in audit: {in_legacy_not_audit.count().execute()}")

Complete sources in audit: 812
Distinct sources in legacy parquet: 1298
In audit (complete) but NOT in legacy: 29
In legacy but NOT complete in audit: 515


In [18]:
missing_from_legacy_df = in_audit_not_legacy.order_by("source_id").to_pandas()
GT(missing_from_legacy_df).tab_header(
    title="Sources Complete in S3 Audit but Missing from Legacy Parquet",
    subtitle=f"{len(missing_from_legacy_df)} sources have annotations.csv with rows but are not in the merged parquet",
)

Sources Complete in S3 Audit but Missing from Legacy Parquet
29 sources have annotations.csv with rows but are not in the merged parquet
source_id
295
372
373
376
554
1645
2112
2615


In [19]:
legacy_not_complete_df = in_legacy_not_audit.order_by("source_id").to_pandas()
GT(legacy_not_complete_df).tab_header(
    title="Sources in Legacy Parquet but NOT Complete in S3 Audit",
    subtitle=f"{len(legacy_not_complete_df)} sources are in merged parquet but fail new completeness check",
)

Sources in Legacy Parquet but NOT Complete in S3 Audit
515 sources are in merged parquet but fail new completeness check
source_id
5457
5458
5459
5460
5461
5465
5467
5468


## Per-source comparison: images and annotations

In [20]:
legacy_by_source = cn_annotations_raw.group_by("source_id").aggregate(
    legacy_n_images=_.image_id.nunique(),
    legacy_n_points=_.count(),
)

audit_subset = s3_audit.select(
    source_id=_.source_id,
    audit_n_images=_.n_unique_images_annotated,
    audit_n_points=_.n_annotations,
    is_complete=_.is_complete,
)

comparison = audit_subset.outer_join(legacy_by_source, "source_id").select(
    source_id=ibis.coalesce(audit_subset.source_id, legacy_by_source.source_id),
    audit_n_images=audit_subset.audit_n_images,
    audit_n_points=audit_subset.audit_n_points,
    legacy_n_images=legacy_by_source.legacy_n_images,
    legacy_n_points=legacy_by_source.legacy_n_points,
    is_complete=audit_subset.is_complete,
)

In [21]:
comparison = comparison.mutate(
    image_diff=_.audit_n_images - _.legacy_n_images,
    point_diff=_.audit_n_points - _.legacy_n_points,
    in_audit_only=_.legacy_n_images.isnull(),
    in_legacy_only=_.audit_n_images.isnull(),
)

In [22]:
comp_summary = {
    "Metric": [
        "Sources in audit only (not in legacy)",
        "Sources in legacy only (not in audit)",
        "Sources in both",
        "Sources with image count mismatch",
        "Sources with point count mismatch",
    ],
    "Count": [
        comparison.filter(_.in_audit_only).count().execute(),
        comparison.filter(_.in_legacy_only).count().execute(),
        comparison.filter(~_.in_audit_only & ~_.in_legacy_only).count().execute(),
        comparison.filter(~_.in_audit_only & ~_.in_legacy_only & (_.image_diff != 0)).count().execute(),
        comparison.filter(~_.in_audit_only & ~_.in_legacy_only & (_.point_diff != 0)).count().execute(),
    ],
}
GT(pd.DataFrame(comp_summary)).tab_header(title="Audit vs Legacy Parquet Comparison Summary")

GT(_tbl_data=                                  Metric  Count
0  Sources in audit only (not in legacy)    211
1  Sources in legacy only (not in audit)      0
2                        Sources in both   1298
3      Sources with image count mismatch    700
4      Sources with point count mismatch    514, _body=<great_tables._gt_data.Body object at 0x1a5fcf770>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1a5fcf9e0>, _spanners=Spanners([]), _heading=Heading(title='Audit vs Legacy Parquet Comparison Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1a5fa2d20>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1a5fa0140>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1a5fa1430>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='heading', type=

In [23]:
totals_comparison = (
    comparison.filter(~_.in_legacy_only)
    .aggregate(
        total_audit_images=_.audit_n_images.sum(),
        total_audit_points=_.audit_n_points.sum(),
        total_legacy_images=_.legacy_n_images.sum(),
        total_legacy_points=_.legacy_n_points.sum(),
    )
    .execute()
)

totals_comparison_df = totals_comparison.T.reset_index()
totals_comparison_df.columns = ["Metric", "Value"]
totals_comparison_df["Value"] = totals_comparison_df["Value"].fillna(0).astype(int)

GT(totals_comparison_df).tab_header(title="Total Counts (Audit vs Legacy)", subtitle="Summed across sources present in audit").fmt_number(
    columns=["Value"], sep_mark=",", decimals=0
)

GT(_tbl_data=                Metric     Value
0   total_audit_images    903805
1   total_audit_points  26812746
2  total_legacy_images    390032
3  total_legacy_points  21326860, _body=<great_tables._gt_data.Body object at 0x1a5fa0b60>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1a5f92b70>, _spanners=Spanners([]), _heading=Heading(title='Total Counts (Audit vs Legacy)', subtitle='Summed across sources present in audit', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1a6043d70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1a6042f90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1a6043f20>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1a60417f0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='heading', type='value', value='center'), heading_title_fon

## Best Sources

- These are sources that were evaluated for quality by Tiela and Coral reef scientists from WCS


In [30]:
best_sources = con.read_csv(BEST_SOURCES, dateformat="%Y-%m-%d", normalize_names=True)
best_sources.head()

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ source_id ┃ source_name                                                             ┃ confirmed_images ┃ total_points_in_confirmed_images ┃ unique_labels_in_confirmed_images ┃ date_created ┃ last_update_in_confirmed_images ┃ classifier_accuracy ┃ classifier_feature_type ┃ classifier_training_images ┃ classifier_date ┃ latitude   ┃ longitude   ┃ affiliation                     ┃ description                                                                      ┃ link                                   ┃ tokeep  ┃ imagequality ┃ coraldiversity ┃ tokeepnotes                                                         ┃ ins3    ┃ inzip   ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩
│ int64     │ string                                                                  │ int64            │ int64                            │ int64                             │ string       │ string                          │ int64               │ string                  │ int64                      │ string          │ float64    │ float64     │ string                          │ string                                                                           │ string                                 │ boolean │ int64        │ int64          │ string                                                              │ boolean │ boolean │
├───────────┼─────────────────────────────────────────────────────────────────────────┼──────────────────┼──────────────────────────────────┼───────────────────────────────────┼──────────────┼─────────────────────────────────┼─────────────────────┼─────────────────────────┼────────────────────────────┼─────────────────┼────────────┼─────────────┼─────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────────┼─────────┼──────────────┼────────────────┼─────────────────────────────────────────────────────────────────────┼─────────┼─────────┤
│      1968 │ Biofouling on the offshore oil platform in the Southwest Gulf of Mexico │              184 │                            41400 │                               138 │ 2020-06-23   │ 2020-09-17                      │                  59 │ vgg16_coralnet_ver1     │                        184 │ 2020-08-26      │  20.164490 │  -91.962410 │ Universidad Auton贸ma de Yucat… │ This project aims to analyze the fauna associated with an oil platform at diffe… │ ]8;id=378835;https://coralnet.ucsd.edu/source/1968/\https://coralnet.ucsd.edu/source/1968/]8;;\ │ True    │            5 │              3 │ One photo (1782679.jpg) has very low diversity                      │ True    │ True    │
│      2855 │ Kaua驶i_Hanalei                                                         │              399 │ 

In [31]:
best_sources_id = best_sources.select(_.source_id).distinct()

## Compare Best Sources to Audit and Legacy Parquet

Compare the curated "best sources" list against:
1. **New audit** - sources with `is_complete = True` (has annotations.csv with n_annotations > 0)
2. **Legacy parquet** - sources present in `coralnet_annotations_30112025.parquet`

In [32]:
best_in_audit_complete = best_sources_id.inner_join(audit_complete, "source_id")
best_in_legacy = best_sources_id.inner_join(legacy_sources, "source_id")
best_not_in_audit_complete = best_sources_id.anti_join(audit_complete, "source_id")
best_not_in_legacy = best_sources_id.anti_join(legacy_sources, "source_id")

print(f"Total best sources: {best_sources_id.count().execute()}")
print(f"Best sources in audit (complete): {best_in_audit_complete.count().execute()}")
print(f"Best sources in legacy parquet: {best_in_legacy.count().execute()}")
print(f"Best sources NOT in audit (complete): {best_not_in_audit_complete.count().execute()}")
print(f"Best sources NOT in legacy parquet: {best_not_in_legacy.count().execute()}")

Total best sources: 192
Best sources in audit (complete): 126
Best sources NOT in audit (complete): 66
Best sources NOT in legacy parquet: 28


In [33]:
best_missing_from_audit = best_not_in_audit_complete.inner_join(
    best_sources.select(_.source_id, _.source_name, _.confirmed_images, _.tokeep, _.tokeepnotes),
    "source_id"
)

In [34]:
best_missing_from_audit

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ source_id ┃ source_name                              ┃ confirmed_images ┃ tokeep  ┃ tokeepnotes                                       ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ int64     │ string                                   │ int64            │ boolean │ string                                            │
├───────────┼──────────────────────────────────────────┼──────────────────┼─────────┼───────────────────────────────────────────────────┤
│      5477 │ Rubble Monitoring                        │             1220 │ False   │ Only one photo has high coral cover (4594851.jpg) │
│      5564 │ Fundacion Huinay Benthic Monitoring 2023 │              334 │ False   │ NA                                                │
│      5752 │ Rocky shore biodiv                       │              658 │ False   │ Photos above water                                │
│      5847 │ CUR Timeseries Orthos_cover              │              211 │ True    │ NA                                                │
│      5942 │ Nexus Israel Mediterranean Monitoring    │              157 │ False   │ All photos of macroalgae                          │
│      6053 │ REEFolution Kenya_Longterm               │              483 │ True    │ NA                                                │
│      6123 │ WoM-AWI                                  │              123 │ False   │ No hard corals                                    │
│      6201 │ Sdot-Yam 10                              │              204 │ False   │ No hard corals                                    │
│      6202 │ Sdot-Yam 45                              │              164 │ False   │ NA                                                │
│      6203 │ Sdot-Yam 25                              │              134 │ False   │ No hard corals                                    │
│         … │ …                                        │                … │ …       │ …                                                 │
└───────────┴──────────────────────────────────────────┴──────────────────┴─────────┴───────────────────────────────────────────────────┘

In [39]:
best_missing_from_legacy = best_not_in_legacy.inner_join(
    best_sources.select(_.source_id, _.source_name, _.confirmed_images, _.tokeep, _.tokeepnotes),
    "source_id"
)

In [40]:
best_missing_from_legacy

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ source_id ┃ source_name                              ┃ confirmed_images ┃ tokeep  ┃ tokeepnotes                                                                      ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ int64     │ string                                   │ int64            │ boolean │ string                                                                           │
├───────────┼──────────────────────────────────────────┼──────────────────┼─────────┼──────────────────────────────────────────────────────────────────────────────────┤
│      3581 │ NFWF Project                             │             4069 │ False   │ Mostly sand and macroalgae                                                       │
│      4559 │ Curacao Coral Reef Assessment 2023       │            12184 │ True    │ One photo dominated by rubble (3846993.jpg), and one photo by sand (4283861.jpg) │
│      4957 │ Pohnpei_Pacific Reef Monitoring Programs │             9011 │ True    │ Two photos are useful: 5619785.jpg & 5723309.jpg                                 │
│      6123 │ WoM-AWI                                  │              123 │ False   │ No hard corals                                                                   │
│      7176 │ KRSP                                     │              401 │ False   │ NA                                                                               │
│      7461 │ Alissa_dominican                         │              100 │ False   │ Only one photo that has identifyable corals                                      │
│       372 │ CREP-REA HAWAII                          │            40478 │ False   │ NA                                                                               │
│       373 │ CREP-REA MARIANAS                        │            26299 │ False   │ It seems that it's taken using a wide lens                                       │
│       554 │ Kahekili                                 │            22395 │ False   │ NA                                                                               │
│      1645 │ Okinawa Coral Reef Conservation2019-20   │             3000 │ True    │ NA                                                                               │
│         … │ …                                        │                … │ …       │ …                                                                                │
└───────────┴──────────────────────────────────────────┴──────────────────┴─────────┴──────────────────────────────────────────────────────────────────────────────────┘

In [41]:
audit_details_for_missing_best = s3_audit.filter(
    _.source_id.isin(best_not_in_audit_complete.source_id)
).select(
    _.source_id,
    _.has_images_folder,
    _.has_annotations_csv,
    _.has_image_list_csv,
    _.n_annotations,
    _.n_unique_images_annotated,
    _.is_complete,
)

In [42]:
audit_details_for_missing_best

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ source_id ┃ has_images_folder ┃ has_annotations_csv ┃ has_image_list_csv ┃ n_annotations ┃ n_unique_images_annotated ┃ is_complete ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int64     │ boolean           │ boolean             │ boolean            │ int64         │ int64                     │ boolean     │
├───────────┼───────────────────┼─────────────────────┼────────────────────┼───────────────┼───────────────────────────┼─────────────┤
│      3058 │ True              │ True                │ True               │             0 │                         0 │ False       │
│      3411 │ True              │ True                │ True               │             0 │                         0 │ False       │
│      5477 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      5564 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      5752 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      5847 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      5942 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      6053 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      6123 │ True              │ False               │ False              │             0 │                         0 │ False       │
│      6201 │ True              │ False               │ False              │             0 │                         0 │ False       │
│         … │ …                 │ …                   │ …                  │             … │                         … │ …           │
└───────────┴───────────────────┴─────────────────────┴────────────────────┴───────────────┴───────────────────────────┴─────────────┘

### Annotation Count Comparison for Best Sources

Compare annotation counts between audit and legacy parquet for best sources present in both.

In [43]:
legacy_agg = cn_annotations_raw.group_by(_.source_id).agg(
    legacy_n_annotations=_.count(),
    legacy_n_images=_.image_id.nunique(),
)

audit_subset = s3_audit.select(
    _.source_id,
    audit_n_annotations=_.n_annotations,
    audit_n_images=_.n_unique_images_annotated,
)

best_comparison = best_sources_id.inner_join(
    audit_subset, "source_id"
).inner_join(
    legacy_agg, "source_id"
).mutate(
    annotation_diff=_.audit_n_annotations - _.legacy_n_annotations,
    image_diff=_.audit_n_images - _.legacy_n_images,
)

In [44]:
sources_with_diff = best_comparison.filter(
    (_.annotation_diff != 0) | (_.image_diff != 0)
)

print(f"Best sources in both audit and legacy: {best_comparison.count().execute()}")
print(f"Sources with annotation/image count differences: {sources_with_diff.count().execute()}")

Best sources in both audit and legacy: 164
Sources with annotation/image count differences: 114


In [38]:
sources_with_diff.order_by(_.annotation_diff.abs().desc())

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ source_id ┃ audit_n_annotations ┃ audit_n_images ┃ legacy_n_annotations ┃ legacy_n_images ┃ annotation_diff ┃ image_diff ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ int64     │ int64               │ int64          │ int64                │ int64           │ int64           │ int64      │
├───────────┼─────────────────────┼────────────────┼──────────────────────┼─────────────────┼─────────────────┼────────────┤
│      6855 │                   0 │              0 │               146900 │            2934 │         -146900 │      -2934 │
│      6868 │                   0 │              0 │               146900 │            2934 │         -146900 │      -2934 │
│      6924 │                   0 │              0 │               146800 │            2934 │         -146800 │      -2934 │
│      6932 │                   0 │              0 │               146800 │            2933 │         -146800 │      -2933 │
│      6733 │                   0 │              0 │               131250 │            2181 │         -131250 │      -2181 │
│      6956 │                   0 │              0 │               127000 │            2060 │         -127000 │      -2060 │
│      6687 │                   0 │              0 │               124400 │            1990 │         -124400 │      -1990 │
│      6218 │                   0 │              0 │               123400 │            1130 │         -123400 │      -1130 │
│      6699 │                   0 │              0 │                92650 │            1482 │          -92650 │      -1482 │
│      6951 │                   0 │              0 │                91500 │            1350 │          -91500 │      -1350 │
│         … │                   … │              … │                    … │               … │               … │          … │
└───────────┴─────────────────────┴────────────────┴──────────────────────┴─────────────────┴─────────────────┴────────────┘